### Search Engine with Tools and Agents

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_community.utilities import ArxivAPIWrapper, WikipediaAPIWrapper
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun

In [3]:
## Calling Wikipedia Wrapper
wiki_wrapper = WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki=WikipediaQueryRun(api_wrapper=wiki_wrapper)
wiki.name

'wikipedia'

In [4]:
## Calling Arxiv Wrapper
arxiv_wrapper = ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv=ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv.name

'arxiv'

In [5]:
## Creating tools
tools=[wiki,arxiv]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/home/prashant/.local/lib/python3.13/site-packages/wikipedia/__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250))]

In [6]:
## Creating Custom Tool
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [7]:
loader=WebBaseLoader("https://docs.langchain.com/langsmith/home")
docx=loader.load()
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)

# Splitting the documents into chunks
docs=text_splitter.split_documents(docx)
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/home', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIToolsPolly AI assistantCLISkillsAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith is a framework-agnostic platform for developing, debugging, and deploying AI agents and LLM applications.'),
 Document(metadata={'source': 'https://docs.langchain.com/langsmith/home', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content=

In [8]:
## Initializing the embedding model
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

## Creating the vectorstore
vectorstore = FAISS.from_documents(documents=docs, embedding=embeddings)

## Creating retriever
retriever = vectorstore.as_retriever()

In [9]:
vectorstore.save_local("langsmith_docs")

In [10]:
from langchain_classic.tools.retriever import create_retriever_tool
retriever_tool=create_retriever_tool(retriever,name="langsmith-search",description="useful for searching any information from the langsmith documentation")
retriever_tool.name

'langsmith-search'

In [11]:
## Updating the tools list
tools.append(retriever_tool)
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/home/prashant/.local/lib/python3.13/site-packages/wikipedia/__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='langsmith-search', description='useful for searching any information from the langsmith documentation', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x7cbfa8829940>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x7cbfa8829120>)]

In [19]:
## Run all tools with Agents and LLM models

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

llm=ChatGroq(model="llama-3.1-8b-instant")

In [20]:
# Prompt template
from langsmith import Client

client = Client()
prompt=client.pull_prompt("hwchase17/openai-functions-agent")
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [31]:
from langchain_classic.agents import create_openai_tools_agent, AgentExecutor

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag

In [33]:
def get_response(query):
    response=agent_executor.invoke({"input": query})
    return response

get_response("What is LangChain?")



> Entering new AgentExecutor chain...

Invoking: `langsmith-search` with `{'query': 'LangChain'}`


Edit this page on GitHub or file an issue.Connect these docs to Claude, VSCode, and more via MCP for real-time answers.Was this page helpful?YesNoCreate an account and API keyNext⌘IDocs by LangChain home pagegithubxlinkedinyoutubeResourcesForumChangelogLangChain AcademyTrust CenterCompanyHomeAboutCareersBloggithubxlinkedinyoutube

It helps you trace requests, evaluate outputs, test prompts, and manage deployments in one place. LangSmith works with any agent stack, whether you’re using an existing framework or building your own. Prototype locally, then move to production with integrated monitoring and evaluation to build more reliable AI agents.
​Get started
Create an accountSign up at smith.langchain.com (no credit card required).
You can log in with Google, GitHub, or email.Create an API keyGo to your Settings page → API Keys → Create API Key.
Copy the key and save it securely.Choose 

{'input': 'What is LangChain?',
 'output': 'LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. It provides a set of tools and libraries that make it easier to build, test, and deploy LLM-based applications. Some potential use cases for LangChain include:\n\n1. Building conversational AI: LangChain can be used to build conversational AI systems that can understand and respond to user input.\n2. Text generation: LangChain can be used to generate text based on a given prompt or topic.\n3. Sentiment analysis: LangChain can be used to analyze the sentiment of a piece of text and determine whether it is positive, negative, or neutral.\n4. Language translation: LangChain can be used to translate text from one language to another.\n5. Summarization: LangChain can be used to summarize long pieces of text into shorter, more digestible versions.\n6. Question answering: LangChain can be used to answer questions based on a give

In [35]:
get_response("What is Machine Learning?")



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'Machine Learning'}`


Page: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks wit
Invoking: `wikipedia` with `{'query': 'Applications of Machine Learning'}`


Page: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks wit
Invoking: `wikipedia` with `{'query': 'Machine Learning algorithms'}`


Page: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks 

{'input': 'What is Machine Learning?',
 'output': 'Machine learning is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without explicit instructions on how to perform a specific task.'}

In [36]:
get_response("What's the paper 1706.03762 about?")



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1706.03762'}`




/home/prashant/.local/lib/python3.13/site-packages/langchain_community/utilities/arxiv.py:102: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  ).results()


Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on c
Invoking: `arxiv` with `{'query': 'Attention Is All You Need'}`
responded: It seems the provided query is not the paper title. Let me try to call the function again with the correct paper title.





/home/prashant/.local/lib/python3.13/site-packages/langchain_community/utilities/arxiv.py:105: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  ).results()


Published: 2021-05-06
Title: Do You Even Need Attention? A Stack of Feed-Forward Layers Does Surprisingly Well on ImageNet
Authors: Luke Melas-Kyriazi
Summary: The strong performance of vision transformers on image classification and other vision tas
Invoking: `arxiv` with `{'query': 'Do You Even Need Attention? A Stack of Feed-Forward Layers Does Surprisingly Well on ImageNet'}`




/home/prashant/.local/lib/python3.13/site-packages/langchain_community/utilities/arxiv.py:105: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  ).results()


Published: 2015-08-17
Title: Observation of the rare $B^0_s\toμ^+μ^-$ decay from the combined analysis of CMS and LHCb data
Authors: The CMS, LHCb Collaborations, :, V. Khachatryan, A. M. Sirunyan, A. Tumasyan, W. Adam, T. Bergauer, M. Dragicevic, J.
Invoking: `arxiv` with `{'query': 'Observation of the rare B^0_s→μ^+μ^- decay from the combined analysis of CMS and LHCb data'}`




/home/prashant/.local/lib/python3.13/site-packages/langchain_community/utilities/arxiv.py:105: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  ).results()


Published: 2015-08-17
Title: Observation of the rare $B^0_s\toμ^+μ^-$ decay from the combined analysis of CMS and LHCb data
Authors: The CMS, LHCb Collaborations, :, V. Khachatryan, A. M. Sirunyan, A. Tumasyan, W. Adam, T. Bergauer, M. Dragicevic, J.
Invoking: `langsmith-search` with `{'query': '1706.03762'}`


Edit this page on GitHub or file an issue.Connect these docs to Claude, VSCode, and more via MCP for real-time answers.Was this page helpful?YesNoCreate an account and API keyNext⌘IDocs by LangChain home pagegithubxlinkedinyoutubeResourcesForumChangelogLangChain AcademyTrust CenterCompanyHomeAboutCareersBloggithubxlinkedinyoutube

​More ways to build
Agent BuilderDesign and deploy AI agents visually with a no-code interface—perfect for rapid prototyping and getting started without writing code.Build an agentPrompt engineeringIterate on prompts with built-in versioning and collaboration to ship improvements faster.Test your promptsPlatform setupUse LangSmith in managed cloud, in 

{'input': "What's the paper 1706.03762 about?",
 'output': 'The paper "1706.03762" is about the transformer model, which is a type of deep learning architecture introduced in the paper "Attention Is All You Need" by Vaswani et al. in 2017. The transformer model is based on self-attention mechanisms and does not use recurrent neural networks (RNNs) or convolutional neural networks (CNNs). It is primarily used for natural language processing tasks such as machine translation, text summarization, and question answering.\n\nThe paper "Do You Even Need Attention?" by Luke Melas-Kyriazi in 2021 explores the idea that a stack of feed-forward layers can perform surprisingly well on image classification tasks, similar to the transformer model. The author argues that the transformer model\'s performance can be replicated using a simpler architecture, which challenges the conventional wisdom that transformer models are necessary for achieving state-of-the-art results.\n\nThe paper "Observation of